# AcadKit — a semester, analysed

Your own semester, from AcadKit's export: attendance habits, how your marks spread, how
attendance and marks relate, what the portal's history looks like, and — once real
results arrive — **how well the app's grade forecasts were calibrated**.

**Data.** In AcadKit: *Settings → Data → Export everything as JSON*. Put the file next to
this notebook (or upload it in Colab) and run all cells. It includes the portal sync history
and the weekly forecast log.

**Privacy.** The export is your personal data. Keep it out of git (`notebooks/*.json` is
ignored) and don't publish it with the notebook.

**Honest scale.** This is one student and one semester: a few hundred attendance rows and a
few dozen marks. The methods here are chosen for small data — rates, medians, rank
correlations, a Beta prior — and every result should be read with that in mind.

In [ ]:
import glob, json, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})

# The newest AcadKit export here (or in Colab's upload folder). Set EXPORT to a path to choose one.
EXPORT = os.environ.get("ACADKIT_EXPORT") or max(
    glob.glob("acadkit-export-*.json") + glob.glob("/content/acadkit-export-*.json"),
    key=os.path.getmtime,
    default=None,
)
assert EXPORT, "No acadkit-export-*.json found — export one from Settings → Data and put it next to this notebook."
data = json.loads(Path(EXPORT).read_text())
print("Loaded", EXPORT, "exported", data.get("exported_at"))

In [ ]:
def frame(key):
    rows = data.get(key) or []
    return pd.DataFrame(rows)

subjects = frame("subjects")
attendance = frame("attendance")
marks = frame("marks")
deadlines = frame("deadlines")
snapshots = frame("portal_snapshots")
history = frame("snapshot_history")
forecasts = frame("forecasts")
archives = frame("archives")

name = dict(zip(subjects.get("id", []), subjects.get("short_name", pd.Series(dtype=str)).fillna(subjects.get("name", ""))))
for df in (attendance, marks, deadlines):
    if "subject_id" in df:
        df["subject"] = df["subject_id"].map(name)

pd.DataFrame(
    {"rows": [len(subjects), len(attendance), len(marks), len(deadlines), len(snapshots), len(history), len(forecasts), len(archives)]},
    index=["subjects", "attendance", "marks", "deadlines", "portal snapshots", "sync history", "forecasts", "archived semesters"],
)

## 1 · Attendance habits

Classes marked by hand (or auto-marked) — not the portal totals, which have no dates.
`holiday` is AcadKit's word for a cancelled class and is left out; On Duty counts as attended.

In [ ]:
if attendance.empty:
    print("No attendance rows in this export.")
else:
    att = attendance[attendance["status"] != "holiday"].copy()
    att["date"] = pd.to_datetime(att["date"])
    att["attended"] = att["status"].isin(["present", "od"]).astype(int)
    att["weekday"] = att["date"].dt.day_name().str[:3]
    att["period"] = att["start_time"].str[:5]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    by_day = att.groupby("weekday")["attended"].mean().reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]).dropna()
    (1 - by_day).plot.bar(ax=axes[0], color="#fb7185", title="Share of classes missed, by weekday")
    by_period = att.groupby("period")["attended"].mean().sort_index()
    (1 - by_period).plot.bar(ax=axes[1], color="#f97316", title="Share missed, by class start time")
    for ax in axes:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
        ax.set_xlabel("")
    plt.tight_layout()

    weekly = att.set_index("date")["attended"].resample("W").mean()
    ax = weekly.plot(marker="o", title="Attendance rate by week", color="#4ade80")
    ax.axhline(0.75, ls="--", color="grey", lw=1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    plt.show()

In [ ]:
if not attendance.empty:
    pivot = att.pivot_table(index="weekday", columns="period", values="attended", aggfunc="mean")
    pivot = pivot.reindex([d for d in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"] if d in pivot.index])
    fig, ax = plt.subplots(figsize=(10, 3.5))
    im = ax.imshow(1 - pivot.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=45)
    ax.set_yticks(range(len(pivot.index)), pivot.index)
    ax.set_title("Where the misses are: weekday × start time (darker = missed more)")
    fig.colorbar(im, ax=ax, format=plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    plt.show()

## 2 · How your marks spread

Each graded component as a share of its maximum. The app's grade odds model these shares
with a Beta distribution around a per-subject ability, with a prior fitted across *all* your
subjects (empirical Bayes) — the numbers below are that prior, estimated the same way
(method of moments).

In [ ]:
if marks.empty:
    print("No marks yet.")
else:
    m = marks[(marks["max_marks"] > 0) & (~marks["is_external"].astype(bool))].copy()
    m["ratio"] = (m["marks_obtained"] / m["max_marks"]).clip(0, 1)
    order = m.groupby("subject")["ratio"].median().sort_values().index
    fig, ax = plt.subplots(figsize=(10, 4))
    # Positional only: matplotlib renamed boxplot's label argument between versions.
    ax.boxplot([m.loc[m["subject"] == s, "ratio"] for s in order])
    ax.set_xticks(range(1, len(order) + 1), order)
    for i, s in enumerate(order, start=1):
        ys = m.loc[m["subject"] == s, "ratio"]
        ax.scatter(np.full(len(ys), i), ys, alpha=0.6, s=18)
    ax.set_title("Component scores by subject (share of max)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    plt.show()

    per_subject = m.groupby("subject")["ratio"]
    mean = float(m["ratio"].clip(0.05, 0.95).mean())
    means = per_subject.mean()
    between = means.var(ddof=1) if len(means) >= 3 else float("nan")
    strength = np.clip(mean * (1 - mean) / between - 1, 2, 30) if between and between > 0 else 4
    resid = m["ratio"] - m["subject"].map(means)
    dof = (per_subject.count() - 1).clip(lower=0).sum()
    within = (resid ** 2).sum() / dof if dof >= 2 else float("nan")
    spread = np.clip(mean * (1 - mean) / within - 1, 3, 60) if within and within > 0 else 8
    print(f"Your typical component share (prior mean): {mean:.1%}")
    print(f"How strongly a subject is pulled toward it (prior strength): {strength:.1f} components' worth")
    print(f"How tightly one subject's components cluster (spread κ): {spread:.1f}")

## 3 · Attendance and marks

Per subject: attendance rate against average component share. With six-odd subjects a
correlation is a hint, not a finding — the rank (Spearman) version is used because it
doesn't pretend the relationship is a straight line.

In [ ]:
if not attendance.empty and not marks.empty:
    a = att.groupby("subject")["attended"].mean().rename("attendance")
    b = m.groupby("subject")["ratio"].mean().rename("marks")
    both = pd.concat([a, b], axis=1).dropna()
    if len(both) >= 3:
        rho = both["attendance"].rank().corr(both["marks"].rank())
        ax = both.plot.scatter(x="attendance", y="marks", s=60, title=f"Attendance vs marks per subject (Spearman ρ = {rho:.2f}, n = {len(both)})")
        for s, row in both.iterrows():
            ax.annotate(s, (row["attendance"], row["marks"]), textcoords="offset points", xytext=(5, 3), fontsize=8)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
        plt.show()
    else:
        print("Need at least three subjects with both attendance and marks.")

## 4 · The portal's history

Every sync is kept (since migration 029). This shows how each subject's totals moved, and
how far behind the portal's figures usually run.

In [ ]:
if history.empty:
    print("No sync history yet — it starts with the first portal sync after migration 029.")
else:
    h = history.copy()
    h["synced_at"] = pd.to_datetime(h["synced_at"], utc=True)
    h["as_of"] = pd.to_datetime(h["as_of"])
    h["pct"] = 100 * (h["conducted"] - h["absent"]) / h["conducted"]
    fig, ax = plt.subplots(figsize=(10, 4))
    for code, g in h.sort_values("synced_at").groupby("subject_code"):
        ax.plot(g["synced_at"], g["pct"], marker="o", label=code)
    ax.axhline(75, ls="--", color="grey", lw=1)
    ax.set_title("Portal attendance % at each sync")
    ax.legend(fontsize=8, ncol=3)
    plt.show()
    per_sync = h.groupby("synced_at")["conducted"].sum().diff().dropna()
    print("Classes added per sync (all subjects): median", per_sync.median(), "· max", per_sync.max())

## 5 · Were the forecasts any good?

Each week the app logs its grade odds: the chance of each subject reaching its target, and
the chance of the semester SGPA reaching yours. When final grades are known (an archived
semester), each forecast can be scored:

* **Brier score** — the mean squared gap between the forecast probability and what
  happened (1 or 0). 0 is perfect; always saying 50% scores 0.25.
* **Calibration** — of the times it said ~70%, did it happen ~70% of the time?

Before results exist, this section shows how the odds moved week by week instead.

In [ ]:
if forecasts.empty:
    print("No forecasts logged yet — the app writes one set per week once migration 029 is in.")
else:
    f = forecasts.copy()
    f["week_start"] = pd.to_datetime(f["week_start"])
    subj = f[f["scope"] != "sgpa"].copy()
    subj["subject"] = subj["scope"].map(name).fillna(subj["scope"])
    ax = subj.pivot_table(index="week_start", columns="subject", values="p_target").plot(marker="o", title="Chance of reaching each target, by week")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend(fontsize=8, ncol=3)
    plt.show()

    # Outcomes: an archived semester's grade per subject code.
    outcomes = {}
    for arch in data.get("archives") or []:
        for row in arch.get("summary") or []:
            outcomes[row["code"]] = row["grade"]
    code_of = dict(zip(subjects.get("id", []), subjects.get("code", [])))
    ladder = ["F", "C", "B", "B+", "A", "A+", "O"]
    subj["actual"] = subj["scope"].map(code_of).map(outcomes)
    scored = subj.dropna(subset=["actual", "target", "p_target"]).copy()
    if scored.empty:
        print("No final grades yet to score against — archive the semester in AcadKit once results are out.")
    else:
        scored["hit"] = scored.apply(lambda r: ladder.index(r["actual"]) >= ladder.index(r["target"]), axis=1).astype(int)
        brier = ((scored["p_target"] - scored["hit"]) ** 2).mean()
        print(f"Brier score over {len(scored)} forecasts: {brier:.3f}  (0 is perfect, 0.25 is a coin flip)")
        bins = pd.cut(scored["p_target"], [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
        cal = scored.groupby(bins, observed=True).agg(said=("p_target", "mean"), happened=("hit", "mean"), n=("hit", "size"))
        ax = cal.plot.scatter(x="said", y="happened", s=cal["n"] * 20, title="Calibration: forecast vs how often it happened")
        ax.plot([0, 1], [0, 1], ls="--", color="grey")
        plt.show()
        display(cal)

## 6 · Deadline load

How the semester's tests and submissions stack up by week — the weeks worth planning around.

In [ ]:
if deadlines.empty:
    print("No deadlines in this export.")
else:
    d = deadlines.copy()
    d["week"] = pd.to_datetime(d["due_date"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None).dt.to_period("W").dt.start_time
    load = d.pivot_table(index="week", columns="type", values="id", aggfunc="count", fill_value=0)
    load.index = load.index.strftime("%d %b")
    load.plot.bar(stacked=True, title="Deadlines per week, by type", figsize=(10, 4))
    plt.xlabel("")
    plt.show()

## Where to take it next

* Model your own attendance: a logistic regression on weekday, start time, subject and
  "days to the next test" (`attendance` has everything needed) — does a test tomorrow
  change whether you go to class today?
* Re-fit the grade model here with a proper hierarchical fit (e.g. PyMC) and compare its
  calibration with the app's empirical-Bayes version.
* After a second semester, compare prior means across semesters — does your baseline move?